# Chapter 10 — The Nearest Neighbor Can Be Wrong

**Book alignment:** Embeddings From First Principles, Chapter 10

**Notebook role:** `ARTIFACT_REPLAY` — reads the frozen experiment artifact(s) this chapter cites and re-derives the numbers quoted in the prose (assertions fail if the artifact drifts).

**Question this notebook isolates:** When the top result is fluent, on-topic, and
geometrically closest — but false — what *kind* of wrong is it, and can a bi-encoder cosine
tell? On RELATE (Wave 1), a general bi-encoder lets a distractor win 0–16% of the time by
type — and an NLI reranker, the obvious "second stage", makes negation and temporal errors
*worse*.

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave, name):
    sub = "wave1/artifacts/v02" if wave == "wave1-v02" else f"{wave}/artifacts"
    return json.loads((EXP / sub / name).read_text())

## 1. Similarity is not equivalence — a toy "aboutness" space

Build vectors where a sentence and its negation share almost all their mass (topic words)
and differ only in a tiny "polarity" coordinate the objective never rewarded.

In [ ]:
def about(topic_vec, polarity):
    v = np.concatenate([topic_vec, [0.05 * polarity]])   # polarity is 1 coordinate out of 33
    return v / np.linalg.norm(v)

topic = rng.standard_normal(32)
query   = about(topic, +1)          # "Dublin IS the capital of Ireland?"
correct = about(topic + rng.standard_normal(32) * 0.15, +1)   # a true paraphrase
negation = about(topic, -1)         # "Dublin is NOT the capital of Ireland"

c_correct  = float(query @ correct)
c_negation = float(query @ negation)
print(f"cos(query, true paraphrase) = {c_correct:.3f}")
print(f"cos(query, negation)        = {c_negation:.3f}")
assert c_negation > c_correct        # the negation is the nearest neighbour
print("\nthe closest vector is the one that flips the claim - 'nearest' was never 'correct'")

## 2. The measured distractor-win-rate by type (Wave 1) — and the reranker backfire

In [ ]:
dw = art("wave1", "distractor-winrate.json")["by_distractor_type"]
print(f"{'distractor type':20} {'bi-encoder':>10} {'+ NLI reranker':>14}")
for t, v in sorted(dw.items(), key=lambda kv: kv[1]["bi_encoder_distractor_win_rate"]):
    print(f"{t:20} {v['bi_encoder_distractor_win_rate']:>10.1%} {v['nli_reranker_distractor_win_rate']:>14.1%}")

# the bi-encoder is not catastrophic on clean restatement queries (0-16% by type) ...
assert max(v["bi_encoder_distractor_win_rate"] for v in dw.values()) < 0.17
# ... but the NLI reranker makes negation and temporal-mismatch WORSE
assert dw["negation"]["nli_reranker_distractor_win_rate"] > dw["negation"]["bi_encoder_distractor_win_rate"]
assert dw["temporal-mismatch"]["nli_reranker_distractor_win_rate"] > dw["temporal-mismatch"]["bi_encoder_distractor_win_rate"]
print('\n"answers the question" and "entails the question" come apart exactly on polarity and time')
print("a reranker helps only if it was trained for the distinction you are missing")

## What we earned

Seven types of near-but-wrong (paraphrase-not-equivalent, negation, entity trap,
same-topic-different-claim, relation swap, temporal mismatch, partial support) — most
invisible to a bi-encoder because the objective encoded *aboutness*, not truth. Two
principles: **similarity is not equivalence**, **retrieval is not verification**. And the
obvious second stage — a generic NLI reranker — raised the negation error from 3.5% to 34%.

**Notebook 11 / Chapter 11** mines the hardest negatives automatically and watches the
comfortable margin collapse.